## Raw Data to Production Dashboards
Raw data sitting in a table isn't an asset until someone can see it. This lab builds the pipeline from raw rows to a live dashboard: medallion layering, declarative transforms, event-driven CDC, a Workspaces-native dashboard, and the path to a deployed Streamlit app.
> **Edition:** Standard+; the BI connection is concept-only, while the runnable payoff is an Altair dashboard in a Workspaces notebook with the Streamlit app deployment path explained.

In this hands-on lab you'll work through 5 sections:

| # | Section | Outcome |
|---|---------|---------|
| 1 | Medallion Architecture Overview (Bronze / Silver / Gold) | understand the layered pattern the rest of the lab builds on |
| 2 | Dynamic Tables — Declarative Pipelines | build a self-refreshing transformation layer and inspect how each refresh mode behaves |
| 3 | Streams + Tasks — Event-Driven CDC | capture changed rows, validate the result, and finalize a multi-step task graph |
| 4 | dbt on Snowflake — Code-First Transformations | see the same pipeline pattern expressed as version-controlled dbt models |
| 5 | BI Tool Connection + Workspaces Dashboard and Streamlit Path | put the finished pipeline in front of a stakeholder as a live dashboard |

### Setup

Create the lab database and warehouse as `SYSADMIN`, and tag this session.

`SYSADMIN` creates them, so `SYSADMIN` owns them — and owns everything the rest of
the lab builds inside them. That is deliberate: the docs say `ACCOUNTADMIN`
"should not be used to create objects in your account," and that `SYSADMIN`
"includes the privileges to create warehouses, databases, and all database objects
(schemas,tables, and so on)." Later cells step up to `ACCOUNTADMIN` only for the
three things that genuinely require an account-level privilege: mounting the
sample-data share, granting `EXECUTE TASK ON ACCOUNT`, and creating the API
integration in Section 4.

> Docs: [Access control best practices](https://docs.snowflake.com/en/user-guide/security-access-control-considerations)


In [ ]:
-- Attribution: tag this session's queries (no privilege needed; role-agnostic)
ALTER SESSION SET QUERY_TAG = '{"name":"raw-data-to-production-dashboards","version":{"major":1,"minor":0}}';

-- Create as the role that should own the result. SYSADMIN owns the lab database,
-- the warehouse, and every object the later sections create inside them, so no
-- ownership repair is ever needed. The docs are explicit on both halves of this:
-- ACCOUNTADMIN "should not be used to create objects in your account", and
-- SYSADMIN "includes the privileges to create warehouses, databases, and all
-- database objects (schemas,tables, and so on)".
-- Source: https://docs.snowflake.com/en/user-guide/security-access-control-considerations
USE ROLE SYSADMIN;

-- IF NOT EXISTS keeps this cell re-runnable. OR REPLACE would drop and recreate
-- the objects, which discards the data and aborts queries running on them.
-- Source: https://docs.snowflake.com/en/sql-reference/sql/create-database
CREATE DATABASE IF NOT EXISTS RAW_DATA_TO_PRODUCTION_DASHBOARDS_HOL;

-- Source: https://docs.snowflake.com/en/sql-reference/sql/create-warehouse
CREATE WAREHOUSE IF NOT EXISTS RAW_DATA_TO_PRODUCTION_DASHBOARDS_WH
  WAREHOUSE_SIZE = XSMALL
  AUTO_SUSPEND = 60
  AUTO_RESUME = TRUE
  INITIALLY_SUSPENDED = TRUE
  COMMENT = 'Dedicated XS compute for this HOL; see the cleanup cell to remove it';

-- Every section from here on runs as SYSADMIN, which owns all three objects.
USE DATABASE RAW_DATA_TO_PRODUCTION_DASHBOARDS_HOL;
USE SCHEMA PUBLIC;
USE WAREHOUSE RAW_DATA_TO_PRODUCTION_DASHBOARDS_WH;


### Reclaim objects from an earlier run (only if Setup fails)

Skip this cell on a first run. It ships disabled.

Uncomment and run it only if Setup or Step 1.1 fails with **"already exists, but
current role has no privileges on it."** That means an earlier run created these
objects under a different role — `SYSADMIN` cannot see them, so it can neither
replace them nor drop them.

Owning a database does not carry ownership of the objects inside it, so each
object type needs its own bulk transfer. `COPY CURRENT GRANTS` is required rather
than optional: the docs note that "A GRANT OWNERSHIP statement fails if existing
outbound privileges on the object are neither revoked nor copied," and Section 5
grants privileges to `BI_SERVICE_ROLE`. Tasks need no manual suspend here —
Snowflake "suspends all tasks in the container automatically if all tasks in a
specified database or schema are transferred to another role."

> Docs: [GRANT OWNERSHIP](https://docs.snowflake.com/en/sql-reference/sql/grant-ownership)


In [ ]:
-- Disabled by default. Uncomment the block below only if Setup or Step 1.1 fails
-- with "already exists, but current role has no privileges on it".
-- Source: https://docs.snowflake.com/en/sql-reference/sql/grant-ownership

-- USE ROLE ACCOUNTADMIN;
-- GRANT OWNERSHIP ON DATABASE RAW_DATA_TO_PRODUCTION_DASHBOARDS_HOL
--   TO ROLE SYSADMIN COPY CURRENT GRANTS;
-- GRANT OWNERSHIP ON SCHEMA RAW_DATA_TO_PRODUCTION_DASHBOARDS_HOL.PUBLIC
--   TO ROLE SYSADMIN COPY CURRENT GRANTS;
-- GRANT OWNERSHIP ON WAREHOUSE RAW_DATA_TO_PRODUCTION_DASHBOARDS_WH
--   TO ROLE SYSADMIN COPY CURRENT GRANTS;
-- GRANT OWNERSHIP ON ALL TABLES IN SCHEMA RAW_DATA_TO_PRODUCTION_DASHBOARDS_HOL.PUBLIC
--   TO ROLE SYSADMIN COPY CURRENT GRANTS;
-- GRANT OWNERSHIP ON ALL DYNAMIC TABLES IN SCHEMA RAW_DATA_TO_PRODUCTION_DASHBOARDS_HOL.PUBLIC
--   TO ROLE SYSADMIN COPY CURRENT GRANTS;
-- GRANT OWNERSHIP ON ALL STREAMS IN SCHEMA RAW_DATA_TO_PRODUCTION_DASHBOARDS_HOL.PUBLIC
--   TO ROLE SYSADMIN COPY CURRENT GRANTS;
-- GRANT OWNERSHIP ON ALL TASKS IN SCHEMA RAW_DATA_TO_PRODUCTION_DASHBOARDS_HOL.PUBLIC
--   TO ROLE SYSADMIN COPY CURRENT GRANTS;
-- USE ROLE SYSADMIN;

SELECT 'Reclaim cell reached. Only the statements uncommented above were run.'
    AS RECLAIM_STATUS;


### Prerequisite — Snowflake sample data

This lab seeds its RAW layer from `SNOWFLAKE_SAMPLE_DATA.TPCH_SF1.ORDERS`, a read-only
database Snowflake shares with your account. It uses no storage, though querying it
consumes credits like any other query. Newer accounts receive it automatically; older
accounts may not.

**Run the next cell as `ACCOUNTADMIN`.** It mounts the share if it is absent, then grants
`IMPORTED PRIVILEGES` to `SYSADMIN` — when a database is created from a share, only the
creating role can read it by default.

Keep the share available for the whole lab: Sections 1 and 2 re-read it when reloading
Bronze and when applying a large change batch.

> Docs: [Use the sample database](https://docs.snowflake.com/en/user-guide/sample-data-using) ·
> [Consume imported data](https://docs.snowflake.com/en/user-guide/data-share-consumers)


In [ ]:
-- Step 0: Mount the sample-data share and make it readable by SYSADMIN.
-- Source: https://docs.snowflake.com/en/user-guide/sample-data-using
-- Source: https://docs.snowflake.com/en/user-guide/data-share-consumers
USE ROLE ACCOUNTADMIN;

-- A share can only be consumed once per account, so guard the mount.
CREATE DATABASE IF NOT EXISTS SNOWFLAKE_SAMPLE_DATA
  FROM SHARE SFC_SAMPLES.SAMPLE_DATA;

-- Section 1 runs as SYSADMIN, which cannot read the share without this grant.
GRANT IMPORTED PRIVILEGES ON DATABASE SNOWFLAKE_SAMPLE_DATA TO ROLE SYSADMIN;

-- Confirm the share resolves and the privilege took effect.
SELECT COUNT(*) AS TPCH_ORDERS_ROWS
FROM SNOWFLAKE_SAMPLE_DATA.TPCH_SF1.ORDERS;


## Section 1 — Medallion Architecture Overview (Bronze / Silver / Gold)

**The problem:** Every raw table, transformation, and dashboard in the same schema means a single
bad migration or type change corrupts your most important reports — with no unmodified
copy to replay from.


The **[medallion architecture](https://docs.snowflake.com/en/user-guide/dynamic-tables/design-patterns)**
organizes a data platform into three isolation layers. This lab uses one mapping throughout:
**Bronze = Raw**, **Silver = Curated**, and **Gold = Analytics**. Bronze preserves a
source-faithful replay point when a downstream transform is wrong. Silver types,
cleans, and deduplicates trusted business entities. Gold pre-aggregates those entities into
a consumption-ready shape. This section seeds Bronze by creating `RAW_ORDERS_TABLE` from Snowflake's provided
`SNOWFLAKE_SAMPLE_DATA.TPCH_SF1.ORDERS` — a Snowflake-provided stand-in for raw-order data.
The sample database is mounted and made readable by the Step 0 prerequisite cell above. Every
subsequent section builds directly on this table and column set.


> **Outcome:** understand the layered pattern the rest of the lab builds on.

## How this lab builds the medallion

Each lab section maps to one layer of the medallion:

| Lab section | Medallion layer | Object created |
|---|---|---|
| **1 — This section** | **Bronze** | `RAW_ORDERS_TABLE` |
| **2 — Dynamic Tables** | **Silver + Gold** | `DT_ORDERS_CURATED`, `DT_ORDERS_ANALYTICS` |
| **3 — Streams + Tasks** | Operational CDC | `ORDERS_STREAM`, task graph, logs |
| **4 — dbt** | **Silver** alternative | `orders_curated.sql` model |
| **5 — BI / dashboard** | **Gold** consumer | `BI_SERVICE_ROLE` + notebook dashboard |

**Bronze is the only layer written to during ingest.** Every layer above reads forward from it — never backward.
Production Bronze is commonly append-only or replace-only so transforms can be re-derived. This disposable
lab later adds and overwrites rows deliberately to demonstrate downstream change processing.

In [ ]:
-- Set session context — SYSADMIN owns all data objects
-- Source: https://docs.snowflake.com/en/sql-reference/sql/use-role
USE ROLE SYSADMIN;
USE DATABASE RAW_DATA_TO_PRODUCTION_DASHBOARDS_HOL;
USE SCHEMA PUBLIC;

### Step 1.1 — Create `RAW_ORDERS_TABLE` (the landing zone)

`RAW_ORDERS_TABLE` is populated with a 50 000-row slice of `SNOWFLAKE_SAMPLE_DATA.TPCH_SF1.ORDERS`,
Snowflake's provided TPCH benchmark dataset. If `SNOWFLAKE_SAMPLE_DATA` is absent, add the
sample database before running this cell. No external stage is required. Columns are renamed to business-friendly names. The full mapping is
in the SQL inline comments so every downstream section knows the exact shape.

| Column | Source column | Type | Notes |
|---|---|---|---|
| `ORDER_ID` | `O_ORDERKEY` | NUMBER | Unique order key |
| `CUSTOMER_ID` | `O_CUSTKEY` | NUMBER | Customer foreign key |
| `STATUS` | `O_ORDERSTATUS` | VARCHAR(20) | `'F'` Fulfilled · `'O'` Open · `'P'` Pending (widened so downstream demo inserts like `'completed'` fit) |
| `ORDER_AMOUNT` | `O_TOTALPRICE` | NUMBER(12,2) | Raw order total in USD |
| `ORDER_DATE` | `O_ORDERDATE` | DATE | Order placement date |
| `PRIORITY` | `O_ORDERPRIORITY` | VARCHAR(15) | `'1-URGENT'` … `'5-LOW'` |
| `CLERK` | `O_CLERK` | VARCHAR(15) | Processing clerk ID |
| `RAW_COMMENT` | `O_COMMENT` | VARCHAR(79) | Free-text; intentionally unclean |

**Why it matters:** This column set is the contract between the RAW layer and everything above it.
Dynamic Tables, Streams+Tasks, dbt, and the notebook dashboard all reference these exact column names — a rename
here surfaces a traceable break in one place instead of silently spreading through the pipeline.

> Docs: [CREATE TABLE — CTAS variant](https://docs.snowflake.com/en/sql-reference/sql/create-table)

In [ ]:
-- SYSADMIN creates it, so SYSADMIN owns it. Stating the role at every creation
-- point keeps ownership correct even if a cell is run on its own or the role
-- picker is changed mid-lab. ACCOUNTADMIN inherits SYSADMIN's objects, so this
-- direction stays readable from either role; the reverse does not.
-- Source: https://docs.snowflake.com/en/user-guide/security-access-control-considerations
USE ROLE SYSADMIN;

-- Step 1.1: Create the RAW landing table via CTAS from SNOWFLAKE_SAMPLE_DATA (TPCH SF1)
-- Source: https://docs.snowflake.com/en/sql-reference/sql/create-table
CREATE OR REPLACE TABLE RAW_ORDERS_TABLE AS
SELECT
    O_ORDERKEY      AS ORDER_ID,       -- unique order identifier
    O_CUSTKEY       AS CUSTOMER_ID,    -- customer foreign key
    O_ORDERSTATUS::VARCHAR(20) AS STATUS,  -- 'F'=Fulfilled 'O'=Open 'P'=Pending; widened for later demo inserts
    O_TOTALPRICE    AS ORDER_AMOUNT,   -- raw order total in USD (unvalidated)
    O_ORDERDATE     AS ORDER_DATE,     -- calendar date of order placement
    O_ORDERPRIORITY AS PRIORITY,       -- '1-URGENT'|'2-HIGH'|'3-MEDIUM'|'4-NOT SPECIFIED'|'5-LOW'
    O_CLERK         AS CLERK,          -- processing clerk ID (e.g. Clerk#000000001)
    O_COMMENT       AS RAW_COMMENT     -- free-text; intentionally left unclean for Silver
FROM SNOWFLAKE_SAMPLE_DATA.TPCH_SF1.ORDERS
LIMIT 50000;

### Step 1.2 — Preview the raw data

Scan the first 10 rows to confirm the table loaded and column names match the contract above.

**You should see:**
- 10 rows with all 8 columns populated
- `STATUS` values: `'F'`, `'O'`, or `'P'`
- `ORDER_AMOUNT` as a decimal (e.g. `173665.47`)
- `RAW_COMMENT` containing raw, un-normalized text — intentional; Silver cleans it

> Docs: [SELECT](https://docs.snowflake.com/en/sql-reference/sql/select)

In [ ]:
-- Step 1.2: Preview the first 10 raw rows
SELECT * FROM RAW_ORDERS_TABLE LIMIT 10;

### Step 1.3 — Verify the load

Confirm the table holds the expected 50 000 rows. If the count is off, re-run Step 1.1 —
`CREATE OR REPLACE` guarantees a clean full reload each time.

**You should see:**
- `ROW_COUNT = 50000`

In [ ]:
-- Step 1.3: Confirm row count
SELECT COUNT(*) AS ROW_COUNT FROM RAW_ORDERS_TABLE;

### Step 1.4 — Inspect the data profile

Before handing data to the next layer, check what's actually there: status distribution,
amount range, and date span. This is the habit that catches bad loads before they propagate.

**Why it matters:** The status breakdown tells Silver what edge cases to handle.
The date span tells Dynamic Tables the time window it will operate over.

**You should see:**
- Three `STATUS` groups: `'F'` (Fulfilled), `'O'` (Open), `'P'` (Pending)
- `MIN_DATE` / `MAX_DATE` spanning 1992 – 1998 (TPCH benchmark range)
- `AVG_AMOUNT` around $150 000 (TPCH scale-factor-1 price distribution)

> Docs: [Aggregate functions](https://docs.snowflake.com/en/sql-reference/functions-aggregation)

In [ ]:
-- Step 1.4: Profile the raw data — status breakdown, amount stats, date span
SELECT
    STATUS,
    COUNT(*)                        AS ORDER_COUNT,
    ROUND(AVG(ORDER_AMOUNT), 2)     AS AVG_AMOUNT,
    ROUND(MIN(ORDER_AMOUNT), 2)     AS MIN_AMOUNT,
    ROUND(MAX(ORDER_AMOUNT), 2)     AS MAX_AMOUNT,
    MIN(ORDER_DATE)                 AS MIN_DATE,
    MAX(ORDER_DATE)                 AS MAX_DATE
FROM RAW_ORDERS_TABLE
GROUP BY STATUS
ORDER BY STATUS;

### Step 1.5 — Idempotency: safe to re-run

`CREATE OR REPLACE TABLE` drops and recreates the table atomically — re-running the ingest
cell never appends duplicates or fails on "table already exists."

**Why it matters:** In production you re-run ingest after schema changes, bad loads, or upstream
corrections. `OR REPLACE` guarantees a clean slate without a manual `DROP TABLE` first.

Re-run the CTAS (Step 1.5a), then confirm the row count stays at 50 000 — not doubled to 100 000 (Step 1.5b):

**You should see after Step 1.5a:**
- `CREATE OR REPLACE TABLE RAW_ORDERS_TABLE` — succeeds silently (no "already exists" error)

**You should see after Step 1.5b:**
- `ROW_COUNT = 50000` — same as Step 1.3, not 100 000

In [ ]:
-- SYSADMIN creates it, so SYSADMIN owns it.
USE ROLE SYSADMIN;

-- Step 1.5a: Re-run the CTAS — CREATE OR REPLACE replaces atomically, never appends
-- Source: https://docs.snowflake.com/en/sql-reference/sql/create-table
CREATE OR REPLACE TABLE RAW_ORDERS_TABLE AS
SELECT
    O_ORDERKEY      AS ORDER_ID,
    O_CUSTKEY       AS CUSTOMER_ID,
    O_ORDERSTATUS::VARCHAR(20) AS STATUS,  -- widened for later demo inserts (matches Step 1.1)
    O_TOTALPRICE    AS ORDER_AMOUNT,
    O_ORDERDATE     AS ORDER_DATE,
    O_ORDERPRIORITY AS PRIORITY,
    O_CLERK         AS CLERK,
    O_COMMENT       AS RAW_COMMENT
FROM SNOWFLAKE_SAMPLE_DATA.TPCH_SF1.ORDERS
LIMIT 50000;

In [ ]:
-- Step 1.5b: Verify count is still 50 000, not 100 000 (idempotency confirmed)
SELECT COUNT(*) AS ROW_COUNT FROM RAW_ORDERS_TABLE;

### Bronze layer complete — foundation laid for the series

`RAW_ORDERS_TABLE` is now the shared baseline every subsequent section builds on:

- **Section 2 (Dynamic Tables)** reads from `RAW_ORDERS_TABLE` and declares
  `DT_ORDERS_CURATED` and `DT_ORDERS_ANALYTICS` as `CREATE DYNAMIC TABLE` statements.
  Snowflake maintains Silver and Gold automatically from that point forward.
- **Section 3 (Streams + Tasks)** creates `ORDERS_STREAM` on `RAW_ORDERS_TABLE` to capture
  new and changed rows, then routes them through a validated task graph.
- **Section 4 (dbt)** expresses the Silver transform as a version-controlled dbt model —
  same logic as the Dynamic Table, but with code review and CI/CD around it.
- **Section 5 (BI / dashboard)** wires a `BI_SERVICE_ROLE` for consumers and renders
  `DT_ORDERS_ANALYTICS` as a Workspaces-native notebook dashboard.

The column set from Step 1.1 (`ORDER_ID`, `CUSTOMER_ID`, `STATUS`, `ORDER_AMOUNT`,
`ORDER_DATE`, `PRIORITY`, `CLERK`, `RAW_COMMENT`) is the contract every section relies on.

### Verify RAW_ORDERS_TABLE in Snowsight (read-only)

[Open in Snowsight](https://app.snowflake.com/_deeplink/#/data/databases/RAW_DATA_TO_PRODUCTION_DASHBOARDS_HOL)

Read-only — confirm what you just built; all objects were created by the SQL above.

> **Apply on your account**
>
> In production, replace the CTAS from SNOWFLAKE_SAMPLE_DATA with a COPY INTO from your
> ingest stage (S3, GCS, or Azure Blob), or a CREATE TABLE AS SELECT from a Kafka topic
> via Snowpipe Streaming. Keep Bronze append-only or replace-only — no UPDATE or
> DELETE. That immutability is what makes it a reliable replay point when upstream data
> quality issues surface weeks or months later.

## Section 2 — Dynamic Tables — Declarative Pipelines

**The problem:** Hand-built pipelines accumulate schedules, dependency code, and refresh logic that teams must maintain as data and requirements change.


A **[Dynamic Table](https://docs.snowflake.com/en/user-guide/dynamic-tables/overview)**
stores the result of a SQL definition and refreshes it toward a
**[TARGET_LAG](https://docs.snowflake.com/en/user-guide/dynamic-tables/target-lag)**
freshness target. This section builds an incremental-compatible Silver table,
an `AUTO` example that resolves to `FULL` once at creation because it uses
unsupported `EXCEPT`, and an `ADAPTIVE` Gold table that normally refreshes
incrementally but can reinitialize when Snowflake's internal heuristics determine
that rebuilding is significantly cheaper.


> **Outcome:** build a self-refreshing transformation layer and inspect how each refresh mode behaves.

### Step 2.1 — Set the working context

Use the lab database and its dedicated transform warehouse.

> Docs: [Dynamic Tables overview](https://docs.snowflake.com/en/user-guide/dynamic-tables/overview)

In [ ]:
-- Step 2.1: set session context
USE ROLE SYSADMIN;
USE DATABASE RAW_DATA_TO_PRODUCTION_DASHBOARDS_HOL;
USE SCHEMA PUBLIC;
USE WAREHOUSE RAW_DATA_TO_PRODUCTION_DASHBOARDS_WH;

### Step 2.2 — Build the incremental Silver table

Create `DT_ORDERS_CURATED` with `REFRESH_MODE = INCREMENTAL`. Its definition
casts columns, normalizes status, filters null keys, and keeps the latest row
per order with `QUALIFY ROW_NUMBER()`. Window functions, including
`ROW_NUMBER`, are generally supported for incremental refresh.

**Why it matters:** Compatible query shape is the first requirement for
processing only changed rows.

> Docs: [Supported queries for Dynamic Tables](https://docs.snowflake.com/en/user-guide/dynamic-tables/supported-queries)

In [ ]:
-- SYSADMIN creates it, so SYSADMIN owns it.
USE ROLE SYSADMIN;

-- Step 2.2: create the incremental-compatible Silver table
-- Source: https://docs.snowflake.com/en/user-guide/dynamic-tables/supported-queries
CREATE OR REPLACE DYNAMIC TABLE DT_ORDERS_CURATED
  TARGET_LAG = '1 minute'
  WAREHOUSE = RAW_DATA_TO_PRODUCTION_DASHBOARDS_WH
  REFRESH_MODE = INCREMENTAL
AS
SELECT
    order_id::NUMBER(38, 0)     AS order_id,
    customer_id::NUMBER(38, 0)  AS customer_id,
    order_date::DATE            AS order_date,
    order_amount::NUMBER(12, 2) AS amount,
    UPPER(TRIM(status))         AS status
FROM RAW_ORDERS_TABLE
WHERE order_id IS NOT NULL
QUALIFY ROW_NUMBER() OVER (
    PARTITION BY order_id
    ORDER BY order_date DESC
) = 1;

### Step 2.3 — Confirm the incremental outcome

`SHOW DYNAMIC TABLES` reports the resolved mode and its creation-time reason.

**You should see:**
- `DT_ORDERS_CURATED`
- `refresh_mode = INCREMENTAL`
- `refresh_mode_reason` is null

> Docs: [SHOW DYNAMIC TABLES](https://docs.snowflake.com/en/sql-reference/sql/show-dynamic-tables)

In [ ]:
-- Step 2.3: expose only the refresh-mode fields used in this lesson
SHOW DYNAMIC TABLES LIKE 'DT_ORDERS_CURATED';
SELECT "name", "refresh_mode", "refresh_mode_reason"
FROM TABLE(RESULT_SCAN(LAST_QUERY_ID()));

### Step 2.4 — Preview Silver

Query the result after the initial refresh completes.

**You should see:** Typed rows, uppercase status values, no null order IDs,
and at most one row per order ID.

In [ ]:
SELECT *
FROM DT_ORDERS_CURATED
ORDER BY order_date DESC
LIMIT 10;

### Step 2.5 — Let `AUTO` resolve once at creation

Create a compact comparison table with `REFRESH_MODE = AUTO`. The definition
uses `EXCEPT`, which is not supported for incremental refresh. Snowflake
therefore creates the table in `FULL` mode.

`AUTO` does not choose a new mode on every refresh. It evaluates the
definition once at creation and locks in `INCREMENTAL` or `FULL`.

**Why it matters:** An unsupported construct is a deterministic way to teach
why `AUTO` resolved to `FULL`; change volume is not a mode-switch trigger.

> Docs: [Dynamic Table refresh modes](https://docs.snowflake.com/en/user-guide/dynamic-tables/refresh-modes)

In [ ]:
-- SYSADMIN creates it, so SYSADMIN owns it.
USE ROLE SYSADMIN;

-- Step 2.5: EXCEPT is unsupported for incremental refresh
-- AUTO succeeds and resolves to FULL once at creation
CREATE OR REPLACE DYNAMIC TABLE DT_ORDERS_AUTO_FULL
  TARGET_LAG = '5 minutes'
  WAREHOUSE = RAW_DATA_TO_PRODUCTION_DASHBOARDS_WH
  REFRESH_MODE = AUTO
AS
SELECT order_id, customer_id, order_date, order_amount, status
FROM RAW_ORDERS_TABLE
EXCEPT
SELECT order_id, customer_id, order_date, order_amount, status
FROM RAW_ORDERS_TABLE
WHERE status IS NULL;

### Step 2.6 — Read the creation-time reason

Inspect the mode immediately after creation.

**You should see:**
- `DT_ORDERS_AUTO_FULL`
- `refresh_mode = FULL`
- `refresh_mode_reason` explaining that the `EXCEPT` set operation is not
  supported for incremental refresh

> Docs: [Dynamic Table refresh modes](https://docs.snowflake.com/en/user-guide/dynamic-tables/refresh-modes)

In [ ]:
-- Step 2.6: AUTO's resolved mode is visible after creation
SHOW DYNAMIC TABLES LIKE 'DT_ORDERS_AUTO_FULL';
SELECT "name", "refresh_mode", "refresh_mode_reason"
FROM TABLE(RESULT_SCAN(LAST_QUERY_ID()));

### Step 2.7 — Build Gold with `ADAPTIVE`

Chain `DT_ORDERS_ANALYTICS` from the incremental Silver table. The definition
uses incrementally supported projection, filtering, and grouping, so it can
run in `ADAPTIVE` mode.

`ADAPTIVE` uses incremental refresh by default. When internal heuristics
determine that an incremental refresh would be significantly more expensive
than rebuilding, Snowflake reinitializes the table and then resumes
incremental refreshes.

**Why it matters:** The mode handles occasional high-change refreshes without
changing the configured mode or permanently falling back to `FULL`.

> Docs: [ADAPTIVE refresh](https://docs.snowflake.com/en/user-guide/dynamic-tables/refresh-modes#adaptive-refresh)

In [ ]:
-- SYSADMIN creates it, so SYSADMIN owns it.
USE ROLE SYSADMIN;

-- Step 2.7: create an incrementally compatible Gold table in ADAPTIVE mode
CREATE OR REPLACE DYNAMIC TABLE DT_ORDERS_ANALYTICS
  TARGET_LAG = '5 minutes'
  WAREHOUSE = RAW_DATA_TO_PRODUCTION_DASHBOARDS_WH
  REFRESH_MODE = ADAPTIVE
AS
SELECT
    DATE_TRUNC('day', order_date) AS order_day,
    status,
    COUNT(*)                      AS order_count,
    SUM(amount)                   AS total_revenue,
    ROUND(AVG(amount), 2)         AS avg_order_value
FROM DT_ORDERS_CURATED
GROUP BY 1, 2;

### Step 2.8 — Confirm the configured `ADAPTIVE` mode

**You should see:**
- `DT_ORDERS_ANALYTICS`
- `refresh_mode = ADAPTIVE`

Large changes can trigger a reinitialization, but they do not switch the
configured mode to `FULL`.

> Docs: [SHOW DYNAMIC TABLES](https://docs.snowflake.com/en/sql-reference/sql/show-dynamic-tables)

In [ ]:
SHOW DYNAMIC TABLES LIKE 'DT_ORDERS_ANALYTICS';
SELECT "name", "refresh_mode", "refresh_mode_reason"
FROM TABLE(RESULT_SCAN(LAST_QUERY_ID()));

### Step 2.9 — Propagate a new Bronze row

Insert a new source row and manually refresh the chain so the lab does not
wait for the target lag.

**Why it matters:** The data changes; the Dynamic Table definitions do not.

> Docs: [ALTER DYNAMIC TABLE — REFRESH](https://docs.snowflake.com/en/sql-reference/sql/alter-dynamic-table)

In [ ]:
INSERT INTO RAW_ORDERS_TABLE
  (order_id, customer_id, order_date, order_amount, status)
VALUES
  (9999, 9999, CURRENT_DATE(), 299.99, 'completed');

In [ ]:
ALTER DYNAMIC TABLE DT_ORDERS_CURATED REFRESH;
ALTER DYNAMIC TABLE DT_ORDERS_ANALYTICS REFRESH;

### Step 2.10 — Confirm the row reached Gold

**You should see:** A `COMPLETED` row for today that includes order 9999 in
the aggregate.

In [ ]:
SELECT *
FROM DT_ORDERS_ANALYTICS
WHERE order_day = CURRENT_DATE()
ORDER BY status;

### Step 2.11 — Establish the small-change baseline

`DYNAMIC_TABLE_REFRESH_HISTORY` reports the action taken for each refresh.
After the single-row insert, the latest successful refresh should normally
use `INCREMENTAL` with a null `reinit_reason`.

**Why it matters:** This gives us a normal-change baseline before we replace
the whole Bronze input.

> Docs: [DYNAMIC_TABLE_REFRESH_HISTORY](https://docs.snowflake.com/en/sql-reference/functions/dynamic_table_refresh_history)

In [ ]:
SELECT
    name,
    refresh_start_time,
    refresh_action,
    reinit_reason,
    state
FROM TABLE(
  SNOWFLAKE.INFORMATION_SCHEMA.DYNAMIC_TABLE_REFRESH_HISTORY(
    NAME_PREFIX => 'RAW_DATA_TO_PRODUCTION_DASHBOARDS_HOL.PUBLIC.DT_ORDERS_ANALYTICS',
    RESULT_LIMIT => 10
  )
)
ORDER BY refresh_start_time DESC;

### Step 2.12 — Apply a large upstream change

Now replace every row in the lab's Bronze table with a repriced 50 000-row
source snapshot. `INSERT OVERWRITE` preserves the table object and grants,
but replaces its rows and change-tracking metadata. Because this lab table
has no declared `RELY` primary key, Snowflake treats the rewritten rows as
changed input.

This is the documented large-change pattern that can make an `ADAPTIVE`
refresh reinitialize. It is intentionally scoped to the disposable HOL
table—do not point this statement at production data.

**You should see:** 50 000 source rows replace the current Bronze contents.

> Docs: [ADAPTIVE refresh](https://docs.snowflake.com/en/user-guide/dynamic-tables/refresh-modes#adaptive-refresh) · [INSERT](https://docs.snowflake.com/en/sql-reference/sql/insert)

In [ ]:
-- Step 2.12a: replace and reprice the complete disposable Bronze snapshot
INSERT OVERWRITE INTO RAW_ORDERS_TABLE (
    order_id,
    customer_id,
    status,
    order_amount,
    order_date,
    priority,
    clerk,
    raw_comment
)
SELECT
    O_ORDERKEY,
    O_CUSTKEY,
    O_ORDERSTATUS::VARCHAR(20),
    ROUND(O_TOTALPRICE * 1.01, 2),
    O_ORDERDATE,
    O_ORDERPRIORITY,
    O_CLERK,
    O_COMMENT
FROM SNOWFLAKE_SAMPLE_DATA.TPCH_SF1.ORDERS
ORDER BY O_ORDERKEY
LIMIT 50000;

In [ ]:
-- Step 2.12b: refresh Silver first, then the ADAPTIVE Gold table
ALTER DYNAMIC TABLE DT_ORDERS_CURATED REFRESH;
ALTER DYNAMIC TABLE DT_ORDERS_ANALYTICS REFRESH;

### Step 2.13 — Inspect the large-change refresh

Query the same history after the all-row overwrite. For an `ADAPTIVE` table,
`refresh_action = 'REINITIALIZE'` identifies the rebuild and `reinit_reason`
explains why it occurred. The configured mode remains `ADAPTIVE`; this is not
`AUTO` changing its creation-time decision.

**You should see:** A recent successful history row alongside the earlier
small-change refresh. `REINITIALIZE` with a populated `reinit_reason` means the
heuristic chose a rebuild. `INCREMENTAL` means it kept incremental processing;
`NO_DATA` means another refresh had already incorporated the change. The
documented `INSERT OVERWRITE` pattern makes reinitialization eligible, but it
does not guarantee the internal heuristic's decision.

**Why it matters:** `SHOW DYNAMIC TABLES` explains the configured or resolved
mode; refresh history explains what one refresh actually did.

> Docs: [DYNAMIC_TABLE_REFRESH_HISTORY](https://docs.snowflake.com/en/sql-reference/functions/dynamic_table_refresh_history)

In [ ]:
SELECT
    name,
    refresh_start_time,
    refresh_action,
    reinit_reason,
    state
FROM TABLE(
  SNOWFLAKE.INFORMATION_SCHEMA.DYNAMIC_TABLE_REFRESH_HISTORY(
    NAME_PREFIX => 'RAW_DATA_TO_PRODUCTION_DASHBOARDS_HOL.PUBLIC.DT_ORDERS_ANALYTICS',
    RESULT_LIMIT => 10
  )
)
ORDER BY refresh_start_time DESC;

### Browse the dynamic tables in Snowsight

[Open in Snowsight](https://app.snowflake.com/_deeplink/#/data/databases/RAW_DATA_TO_PRODUCTION_DASHBOARDS_HOL)

Read-only — confirm what you just built; all objects were created by the SQL above.

> **Apply on your account**
>
> **Apply on your account:**
> - Use Bronze for as-ingested data, Silver for clean trusted rows, and Gold for
>   consumption-ready aggregates.
> - Treat `AUTO` as a creation-time recommendation. Inspect `name`, `refresh_mode`,
>   and `refresh_mode_reason`, then choose an explicit production mode.
> - Use `INCREMENTAL` or `ADAPTIVE` only with incrementally supported definitions.
>   `ROW_NUMBER` is supported; `EXCEPT`, `INTERSECT`, and `MINUS` are not.
> - A large change can make incremental work expensive, but it does not switch a
>   configured `AUTO` or `INCREMENTAL` table to `FULL`. `ADAPTIVE` can reinitialize
>   under internal heuristics and then resumes incremental refreshes.
> - This lab demonstrates the large-change pattern with a disposable
>   `INSERT OVERWRITE`, then compares its refresh action with the earlier one-row
>   change. Never reuse the overwrite statement against a production source table.
> - Monitor `refresh_action` and `reinit_reason` in
>   `DYNAMIC_TABLE_REFRESH_HISTORY`; the observed history—not an assumed percentage
>   threshold—is the source of truth.

## Section 3 — Streams + Tasks — Event-Driven CDC

**The problem:** A single scheduled SQL statement can move data, but it does not validate the
result, pass runtime context to downstream work, or record graph completion.


**[Change Data Capture](https://docs.snowflake.com/en/user-guide/streams-intro)**
(CDC) exposes changed rows through a **stream**, while a **[task graph](https://docs.snowflake.com/en/user-guide/tasks-graphs)**
coordinates a root transform, a validation child, and a finalizer. Snowflake
Scripting lets the root set a row-count return value, the child validate it, and
the finalizer write a completion record after the graph finishes.


> **Outcome:** capture changed rows, validate the result, and finalize a multi-step task graph.

### Step 3.1 — Grant task execution and set context

Grant `EXECUTE TASK` to the graph owner, then return to `SYSADMIN` for
data-object work.

**Run as:** ACCOUNTADMIN for the grant, then SYSADMIN.

`EXECUTE TASK` is an account-level (global) privilege, and a role can only grant
privileges it holds and can grant. `SECURITYADMIN` holds `MANAGE GRANTS` but does
not hold `EXECUTE TASK`, so it cannot grant it. Worse, the statement does not fail
loudly: it returns the warning `Grant not executed: Insufficient privileges.`,
grants nothing, and the failure only surfaces later when `ALTER TASK ... RESUME`
cannot start the task.

**Why it matters:** "Which role grants this?" is a different question from "which
role owns this?" `SECURITYADMIN` is the right role for granting a *role to another
role*; account-level privileges come from `ACCOUNTADMIN`.

> Docs: [Introduction to tasks](https://docs.snowflake.com/en/user-guide/tasks-intro) ·
> [GRANT privileges TO ROLE](https://docs.snowflake.com/en/sql-reference/sql/grant-privilege)

In [ ]:
-- ACCOUNTADMIN, not SECURITYADMIN: account-level (global) privileges must be
-- granted by a role that holds them. SECURITYADMIN has MANAGE GRANTS but does not
-- hold EXECUTE TASK, so it cannot pass it on -- the statement returns the warning
-- "Grant not executed: Insufficient privileges." and silently grants nothing.
-- Source: https://docs.snowflake.com/en/user-guide/tasks-intro
USE ROLE ACCOUNTADMIN;
GRANT EXECUTE TASK ON ACCOUNT TO ROLE SYSADMIN;

USE ROLE SYSADMIN;
USE DATABASE RAW_DATA_TO_PRODUCTION_DASHBOARDS_HOL;
USE SCHEMA PUBLIC;
USE WAREHOUSE RAW_DATA_TO_PRODUCTION_DASHBOARDS_WH;

### Step 3.2 — Create the change stream

Attach `ORDERS_STREAM` to the Bronze source before generating new changes.
A plain `SELECT` can inspect the stream without advancing its offset.

**You should see:** One non-stale stream.

> Docs: [CREATE STREAM](https://docs.snowflake.com/en/sql-reference/sql/create-stream) · [Streams introduction](https://docs.snowflake.com/en/user-guide/streams-intro)

In [ ]:
-- SYSADMIN creates it, so SYSADMIN owns it.
USE ROLE SYSADMIN;

CREATE OR REPLACE STREAM ORDERS_STREAM
  ON TABLE RAW_ORDERS_TABLE
  COMMENT = 'CDC stream for RAW_ORDERS_TABLE';

SHOW STREAMS LIKE 'ORDERS_STREAM';
SELECT "name", "stale"
FROM TABLE(RESULT_SCAN(LAST_QUERY_ID()));

### Step 3.3 — Generate two change events

Insert two orders after the stream's creation offset.

> Docs: [Stream columns](https://docs.snowflake.com/en/user-guide/streams-intro#stream-columns)

In [ ]:
INSERT INTO RAW_ORDERS_TABLE
  (order_id, customer_id, order_date, order_amount, status)
VALUES
  (9001, 201, CURRENT_DATE(), 149.99, 'PENDING'),
  (9002, 202, CURRENT_DATE(), 299.00, 'PROCESSING');

### Step 3.4 — Inspect without consuming

Stream metadata identifies the action, update pairing, and stable row ID.
Querying alone leaves the offset unchanged.

**You should see:** Two `INSERT` rows with `is_update_op = FALSE`.

> Docs: [Stream columns](https://docs.snowflake.com/en/user-guide/streams-intro#stream-columns)

In [ ]:
SELECT
    order_id,
    order_amount,
    status,
    METADATA$ACTION   AS cdc_action,
    METADATA$ISUPDATE AS is_update_op,
    METADATA$ROW_ID   AS row_id
FROM ORDERS_STREAM
ORDER BY order_id;

### Step 3.5 — Create data and run logs

The CDC log retains consumed rows. The run log gives the finalizer a durable
place to record graph completion.

In [ ]:
-- SYSADMIN creates it, so SYSADMIN owns it.
USE ROLE SYSADMIN;

CREATE OR REPLACE TABLE ORDERS_CDC_LOG (
    log_id      NUMBER AUTOINCREMENT,
    order_id    NUMBER,
    customer_id NUMBER,
    amount      NUMBER(12, 2),
    status      VARCHAR,
    cdc_action  VARCHAR,
    is_update   BOOLEAN,
    logged_at   TIMESTAMP_LTZ DEFAULT CURRENT_TIMESTAMP()
);

CREATE OR REPLACE TABLE ORDERS_CDC_RUN_LOG (
    root_task_name VARCHAR,
    scheduled_at   TIMESTAMP_LTZ,
    finalized_at   TIMESTAMP_LTZ
);

### Step 3.6 — Build a three-stage task graph

The root consumes the stream and sets its inserted-row count with
`SYSTEM$SET_RETURN_VALUE`. The validation child retrieves that value with
`SYSTEM$GET_PREDECESSOR_RETURN_VALUE` and raises an exception when no rows
were loaded. The finalizer records completion after the graph's other tasks
finish.

**Why it matters:** A production pipeline needs data movement, validation,
and lifecycle handling—not only one SQL transform.

> Docs: [Task graphs and return values](https://docs.snowflake.com/en/user-guide/tasks-graphs) · [Snowflake Scripting DML status](https://docs.snowflake.com/en/developer-guide/snowflake-scripting/dml-status)

In [ ]:
-- SYSADMIN creates it, so SYSADMIN owns it.
USE ROLE SYSADMIN;

-- Dollar-sign delimiters make Snowflake Scripting parse correctly in
-- Snowflake CLI. See: https://docs.snowflake.com/en/developer-guide/snowflake-scripting/running-examples
-- Root: checks stream metadata before starting. SYSTEM$STREAM_HAS_DATA
-- avoids false negatives but can return a false positive.
CREATE OR REPLACE TASK ORDERS_CDC_TASK
  WAREHOUSE = RAW_DATA_TO_PRODUCTION_DASHBOARDS_WH
  WHEN SYSTEM$STREAM_HAS_DATA('ORDERS_STREAM')
AS
EXECUTE IMMEDIATE $$
DECLARE
  rows_loaded NUMBER;
  result_string VARCHAR;
BEGIN
  INSERT INTO ORDERS_CDC_LOG
    (order_id, customer_id, amount, status, cdc_action, is_update)
  SELECT
      order_id,
      customer_id,
      order_amount,
      status,
      METADATA$ACTION,
      METADATA$ISUPDATE
  FROM ORDERS_STREAM
  WHERE METADATA$ACTION = 'INSERT';

  rows_loaded := SQLROWCOUNT;
  result_string := TO_VARCHAR(rows_loaded);
  CALL SYSTEM$SET_RETURN_VALUE(:result_string);
END;
$$;

-- Child: use the root's return value as a runnable quality gate
CREATE OR REPLACE TASK ORDERS_CDC_VALIDATE_TASK
  WAREHOUSE = RAW_DATA_TO_PRODUCTION_DASHBOARDS_WH
  AFTER ORDERS_CDC_TASK
AS
EXECUTE IMMEDIATE $$
DECLARE
  rows_loaded NUMBER;
  validation_failed EXCEPTION (-20001, 'CDC validation failed: no rows loaded.');
  result_string VARCHAR;
BEGIN
  rows_loaded := (
    SELECT TO_NUMBER(
      SYSTEM$GET_PREDECESSOR_RETURN_VALUE('ORDERS_CDC_TASK')
    )
  );

  IF (rows_loaded < 1) THEN
    RAISE validation_failed;
  END IF;

  result_string := 'validated ' || TO_VARCHAR(:rows_loaded) || ' loaded rows';
  CALL SYSTEM$SET_RETURN_VALUE(:result_string);
END;
$$;

-- Finalizer: runs after the graph completes or fails
CREATE OR REPLACE TASK ORDERS_CDC_FINALIZER_TASK
  WAREHOUSE = RAW_DATA_TO_PRODUCTION_DASHBOARDS_WH
  FINALIZE = ORDERS_CDC_TASK
AS
EXECUTE IMMEDIATE $$
BEGIN
  INSERT INTO ORDERS_CDC_RUN_LOG
    (root_task_name, scheduled_at, finalized_at)
  SELECT
    SYSTEM$TASK_RUNTIME_INFO('CURRENT_ROOT_TASK_NAME'),
    SYSTEM$TASK_RUNTIME_INFO(
      'CURRENT_TASK_GRAPH_ORIGINAL_SCHEDULED_TIMESTAMP'
    )::TIMESTAMP_LTZ,
    CURRENT_TIMESTAMP();

  CALL SYSTEM$SET_RETURN_VALUE('graph finalized');
END;
$$;

### Walkthrough — add an email notification

Email is intentionally not runnable in this HOL. `SYSTEM$SEND_EMAIL` requires
an email notification integration and validated recipients. In a production
graph, add the call to the finalizer after provisioning that integration.

> **Walkthrough only:** Do not run this cell until your account administrator
> has created the integration and approved the recipient.

> Docs: [SYSTEM$SEND_EMAIL](https://docs.snowflake.com/en/user-guide/notifications/email-stored-procedures)

In [ ]:
/* WALKTHROUGH ONLY — DO NOT RUN
-- Requires an email notification integration and validated recipient.
CALL SYSTEM$SEND_EMAIL(
  'YOUR_EMAIL_NOTIFICATION_INTEGRATION',
  'data-team@example.com',
  'Orders CDC graph finished',
  'Review ORDERS_CDC_RUN_LOG and task history for this graph run.'
);
END WALKTHROUGH */
SELECT 'Walkthrough only — no email sent.' AS walkthrough_status;

### Step 3.7 — Enable the children and run the graph

For a manual test, resume the validation child and finalizer while leaving
the root suspended, then execute the root directly. This avoids starting a
triggered run while the graph version is still being established.

`SYSTEM$STREAM_HAS_DATA` checks stream metadata before the root starts. It is
designed to avoid false negatives, but it can return a false positive, so the
task body and validation still need to handle a zero-row result safely.

**You should see:** The graph starts asynchronously. Run the bounded wait cell
next before querying its outputs.

> Docs: [Run a task graph](https://docs.snowflake.com/en/user-guide/tasks-graphs#run-or-schedule-tasks-in-a-task-graph) · [SYSTEM$STREAM_HAS_DATA](https://docs.snowflake.com/en/sql-reference/functions/system_stream_has_data)

In [ ]:
ALTER TASK ORDERS_CDC_VALIDATE_TASK RESUME;
ALTER TASK ORDERS_CDC_FINALIZER_TASK RESUME;
SET HOL_GRAPH_STARTED_AT = CURRENT_TIMESTAMP();
EXECUTE TASK ORDERS_CDC_TASK;

### Step 3.8 — Wait for graph completion

`EXECUTE TASK` starts the graph asynchronously. Poll completed graph history for
up to 60 seconds before reading the output tables, and fail clearly if this
invocation does not reach `SUCCEEDED`.

**You should see:** `Task graph completed successfully.`

> Docs: [COMPLETE_TASK_GRAPHS](https://docs.snowflake.com/en/sql-reference/functions/complete_task_graphs) · [SYSTEM$WAIT](https://docs.snowflake.com/en/sql-reference/functions/system_wait)

In [ ]:
-- Step 3.8: wait up to 60 seconds for the asynchronous graph to finish.
-- Filters on the start time captured in Step 3.7, so an older run cannot match.
-- Run Step 3.7 first in the same session.
-- Source: https://docs.snowflake.com/en/sql-reference/functions/system_wait
-- Source: https://docs.snowflake.com/en/sql-reference/functions/complete_task_graphs
EXECUTE IMMEDIATE $$
DECLARE
  attempts NUMBER DEFAULT 0;
  graph_state VARCHAR DEFAULT NULL;
  graph_timeout EXCEPTION (-20002, 'Task graph did not finish within 60 seconds.');
  graph_failed EXCEPTION (-20003, 'Task graph finished in a non-success state.');
BEGIN
  WHILE (graph_state IS NULL AND attempts < 30) DO
    SELECT MAX(state)
      INTO :graph_state
    -- Fully qualified: Information Schema table functions need the database
    -- either in the name or in the session context.
    FROM TABLE(
      RAW_DATA_TO_PRODUCTION_DASHBOARDS_HOL.INFORMATION_SCHEMA.COMPLETE_TASK_GRAPHS(
        RESULT_LIMIT => 10,
        ROOT_TASK_NAME => 'ORDERS_CDC_TASK'
      )
    )
    WHERE scheduled_time >= $HOL_GRAPH_STARTED_AT;

    IF (graph_state IS NULL) THEN
      CALL SYSTEM$WAIT(2, 'SECONDS');
    END IF;

    attempts := attempts + 1;
  END WHILE;

  IF (graph_state IS NULL) THEN
    RAISE graph_timeout;
  END IF;

  IF (graph_state <> 'SUCCEEDED') THEN
    RAISE graph_failed;
  END IF;

  RETURN 'Task graph completed successfully.';
END;
$$;

### Step 3.9 — Verify data, offset, and finalization

Check each visible outcome separately.

**You should see:**
- Orders 9001 and 9002 in `ORDERS_CDC_LOG`
- Zero rows left in `ORDERS_STREAM`
- One completion row in `ORDERS_CDC_RUN_LOG`

> Docs: [TASK_HISTORY](https://docs.snowflake.com/en/sql-reference/functions/task_history)

In [ ]:
SELECT *
FROM ORDERS_CDC_LOG
ORDER BY logged_at DESC, log_id;

In [ ]:
SELECT COUNT(*) AS rows_left_in_stream
FROM ORDERS_STREAM;

In [ ]:
SELECT *
FROM ORDERS_CDC_RUN_LOG
ORDER BY finalized_at DESC;

### Step 3.10 — Suspend the root after the demo

Stop future triggered runs while preserving the graph for inspection.

> Docs: [ALTER TASK](https://docs.snowflake.com/en/sql-reference/sql/alter-task)

In [ ]:
ALTER TASK ORDERS_CDC_TASK SUSPEND;

> **Apply on your account**
>
> **Apply on your account:**
> - Keep all tasks in a graph under one owner and in one database and schema.
> - Use return values for compact runtime context, not as a substitute for durable
>   operational logs.
> - Make validation fail the graph when the data contract is not met.
> - Use a finalizer for cleanup, audit records, or notifications after the rest of
>   the graph completes.
> - Provision and grant `USAGE` on a notification integration before adding
>   `SYSTEM$SEND_EMAIL`; the HOL leaves that account-specific dependency as a
>   walkthrough.
> - Treat `SYSTEM$STREAM_HAS_DATA` as a metadata guard, not proof that rows exist.
>   It can return false positives, so handle a zero-row DML result explicitly.
> - Do not describe stream consumption as end-to-end exactly-once delivery.
>   Design downstream writes to be idempotent and monitor task history.

## Section 4 — dbt on Snowflake — Code-First Transformations

**The problem:** Dynamic Tables and Streams+Tasks handle orchestration entirely inside Snowflake, but teams with an existing dbt codebase need their transformation logic version-controlled, peer-reviewed in pull requests, and verified with dbt tests — without standing up an external scheduler just to run `dbt run`.


**[dbt Projects on Snowflake](https://docs.snowflake.com/en/user-guide/data-engineering/dbt-projects-on-snowflake)** deploys a **[dbt Core](https://docs.getdbt.com/docs/introduction)** (data build tool) project as a schema-level Snowflake object. Once deployed, you run an explicit dbt command such as `run`, `test`, or `build` through `EXECUTE DBT PROJECT` SQL or the `snow dbt execute` CLI. Snowflake manages the dbt runtime, and a native **[Task](https://docs.snowflake.com/en/sql-reference/sql/create-task)** can schedule execution without requiring an external orchestrator. The output is the same curated orders table built with **[Dynamic Tables](https://docs.snowflake.com/en/user-guide/dynamic-tables/overview)** in the previous section, expressed as version-controlled `.sql` model files with dbt tests and generated documentation.

> **Walkthrough only:** this section requires an external git repository containing a dbt Core project and a Snowflake git repository integration pointing to it. Use these cells as a reference when you wire it up yourself.


> **Outcome:** see the same pipeline pattern expressed as version-controlled dbt models.

### When to choose dbt vs dynamic tables vs streams and tasks

All three approaches in this webinar consume **RAW_ORDERS_TABLE** and produce curated output. The right tool depends on your team's workflow:

| Approach | Best for | What you gain |
|---|---|---|
| **Dynamic Tables** | Pure SQL teams, no external tooling | Declarative, refresh managed by Snowflake |
| **Streams + Tasks** | Row-level CDC, event-driven triggers | Reacts to new/changed rows immediately |
| **dbt on Snowflake** | Teams with an existing dbt codebase | Version control, `dbt test`, generated docs, large ecosystem |

**Why it matters:** if your team already writes dbt models and runs `dbt test` in CI, deploying the project as a native Snowflake object gives you the same dbt authoring experience with Snowflake as the executor — no separate scheduler bill, no extra operations surface to maintain.

> Docs: [dbt Projects on Snowflake overview](https://docs.snowflake.com/en/user-guide/data-engineering/dbt-projects-on-snowflake)

### Step 4.1 — Connect Snowflake to your Git repository

Before creating a dbt project object, Snowflake needs two things: an **[API integration](https://docs.snowflake.com/en/sql-reference/sql/create-api-integration)** that authenticates outbound HTTPS connections to your git provider (GitHub, GitLab, Bitbucket), and a **[git repository integration](https://docs.snowflake.com/en/sql-reference/sql/create-git-repository)** that mirrors the remote repo as a Snowflake stage. Once both exist, your repo's branches and files are reachable at the path `@db.schema.repo_name/branches/branch_name/...` — that path is what `CREATE DBT PROJECT` uses in the next step.

**Why it matters:** the git repository stage is the deployment bridge between Snowflake and your codebase. Snowflake copies the selected project files into the dbt project object; execution uses that deployed copy rather than reading the live branch on every run.

**Run as:** ACCOUNTADMIN for the API integration (account-level object); SYSADMIN for the git repository object (schema-level, CREATE GIT REPOSITORY privilege).

> Docs: [CREATE API INTEGRATION](https://docs.snowflake.com/en/sql-reference/sql/create-api-integration) · [CREATE GIT REPOSITORY](https://docs.snowflake.com/en/sql-reference/sql/create-git-repository)

In [ ]:
/* WALKTHROUGH ONLY — DO NOT RUN
The URLs below are placeholders. Copy these statements into a worksheet and
substitute your own org and repository.

-- Step 4.1a: Create the API integration (account-level; requires ACCOUNTADMIN)
-- Source: https://docs.snowflake.com/en/sql-reference/sql/create-api-integration
USE ROLE ACCOUNTADMIN;

CREATE OR REPLACE API INTEGRATION dbt_git_api_integration
  API_PROVIDER = git_https_api                               -- GitHub / GitLab / Bitbucket over HTTPS
  API_ALLOWED_PREFIXES = ('https://github.com/your-org/')    -- restrict to your org
  ENABLED = TRUE;
  -- For private repos, also add:
  -- ALLOWED_AUTHENTICATION_SECRETS = (db.schema.git_secret)

-- Step 4.1b: Let SYSADMIN use the integration. CREATE GIT REPOSITORY requires USAGE
-- on the API integration, so creating it as ACCOUNTADMIN is not enough on its own.
GRANT USAGE ON INTEGRATION dbt_git_api_integration TO ROLE SYSADMIN;

-- Step 4.1c: Create the git repository object (schema-level; SYSADMIN with CREATE GIT REPOSITORY)
-- Source: https://docs.snowflake.com/en/sql-reference/sql/create-git-repository
USE ROLE SYSADMIN;

CREATE OR REPLACE GIT REPOSITORY RAW_DATA_TO_PRODUCTION_DASHBOARDS_HOL.PUBLIC.dbt_orders_repo
  API_INTEGRATION = dbt_git_api_integration
  ORIGIN = 'https://github.com/your-org/your-dbt-project.git';
  -- For private repos, also add:
  -- GIT_CREDENTIALS = db.schema.git_secret

-- Verify Snowflake can reach the repo and list available branches
-- (the repo exposes a stage at @RAW_DATA_TO_PRODUCTION_DASHBOARDS_HOL.PUBLIC.dbt_orders_repo)
SHOW GIT BRANCHES IN GIT REPOSITORY RAW_DATA_TO_PRODUCTION_DASHBOARDS_HOL.PUBLIC.dbt_orders_repo;
END WALKTHROUGH */
SELECT 'Walkthrough only — no integration or repository created.' AS walkthrough_status;


### Step 4.2 — Create the dbt project object

`CREATE DBT PROJECT` copies your project files into a **[dbt project object](https://docs.snowflake.com/en/user-guide/data-engineering/dbt-projects-on-snowflake-understanding-dbt-project-objects)**. You can point it at a branch inside a git repository stage using `FROM '@db.schema.repo/branches/branch'`. The deployed copy records *which code to run* and does not change merely because the remote branch changes.

Two key distinctions from other Snowflake objects:

**Target database/schema come from `profiles.yml`, not from `CREATE DBT PROJECT`.** The `DEFAULT_TARGET` parameter names the target block to use by default. The `account` and `user` fields can be placeholder strings because execution uses the Snowflake session context. The execution roles still need sufficient privileges on the target database, schema, and warehouse; dbt can create the target schema when those privileges permit it.

**`EXECUTE DBT PROJECT` runs the deployed object, not the live git branch.** During the 2026_06 behavior-change transition, accounts can expose either legacy numbered versions or the newer single mutable `live` version. On a live-version object, fetch the repository and use `ALTER DBT PROJECT ... DEPLOY FROM ...`; follow the numbered-version documentation if your account has not migrated yet.

**Why it matters:** unlike local `dbt run`, the project object is a native Snowflake citizen: grantable with RBAC, schedulable with Tasks, and visible in Query History with full run logs.

> Docs: [CREATE DBT PROJECT](https://docs.snowflake.com/en/sql-reference/sql/create-dbt-project) · [Understanding dbt project objects](https://docs.snowflake.com/en/user-guide/data-engineering/dbt-projects-on-snowflake-understanding-dbt-project-objects)

In [ ]:
/* WALKTHROUGH ONLY — DO NOT RUN
-- Step 4.2: Create the dbt project object from the git repository stage
-- Source: https://docs.snowflake.com/en/sql-reference/sql/create-dbt-project
USE ROLE SYSADMIN;

-- The GIT REPOSITORY from Step 4.1 exposes a stage at @db.schema.repo_name.
-- FROM points at the selected branch path and deploys those project files.
CREATE DBT PROJECT RAW_DATA_TO_PRODUCTION_DASHBOARDS_HOL.PUBLIC.dbt_orders_project
  FROM '@RAW_DATA_TO_PRODUCTION_DASHBOARDS_HOL.PUBLIC.dbt_orders_repo/branches/main'
  DEFAULT_TARGET = 'prod'             -- matches a target name in profiles.yml inside the repo
  COMMENT = 'Orders transformation pipeline: curated layer from raw orders';

-- Confirm the project object was registered.
SHOW DBT PROJECTS;
END WALKTHROUGH */
SELECT 'Walkthrough only — no dbt project created.' AS walkthrough_status;


### Step 4.3 — What a dbt model file looks like

A dbt model is a `.sql` file inside the `models/` directory of your git repo. Snowflake reads it from the deployed project object at execution time and materializes it as a table or view — the `config(...)` macro at the top of the file controls how. The model below produces the same curated orders output as the Dynamic Table from Section 2: same status filter, same `order_amount` column, expressed as a dbt model using the `source()` macro to reference **RAW_ORDERS_TABLE** by its source name.

```sql
-- models/orders_curated.sql  (lives in your git repo; NOT executed directly in Snowflake)

{{
  config(materialized='table')
}}

SELECT
    order_id,
    customer_id,
    order_date,
    status,
    order_amount          AS order_amount_usd,   -- same source column as Section 2's Dynamic Table
    priority,
    clerk
FROM {{
  source('raw', 'RAW_ORDERS_TABLE')
}}             -- source() references the Bronze layer
WHERE status != 'CANCELLED'
  AND order_id IS NOT NULL
```

**Why it matters:** this is identical transformation logic to Section 2's Dynamic Table. What changes is the delivery: the model lives in git, gets reviewed in a pull request, and `dbt test` can assert `not_null` on `order_id` or `accepted_values` on `status` before a release or downstream promotion completes. The compiled model SQL is the same SELECT pattern you wrote in Section 2.

> Docs: [dbt Projects on Snowflake overview](https://docs.snowflake.com/en/user-guide/data-engineering/dbt-projects-on-snowflake) · [dbt model configurations](https://docs.getdbt.com/reference/model-configs)

### Step 4.4 — Execute the project

You trigger the deployed project two ways: in SQL with `EXECUTE DBT PROJECT`, or from a terminal with the `snow dbt execute` CLI. In both cases, the command you choose controls the work: `run` executes selected models, `test` executes tests, and `build` runs and tests selected resources in directed-acyclic-graph order.

The `ARGS` parameter accepts a supported dbt command and its flags. `ARGS = 'run --select orders_curated --target prod'` runs that model but does not run its tests. Use `ARGS = 'build --target prod'` when the same invocation must include model execution and tests. `DEFAULT_TARGET` supplies the target when `--target` is omitted.

**Why it matters:** the SQL form is exactly what you place inside a Task for scheduling (Step 4.5). The CLI form is what you call from GitHub Actions, local dev, or manual re-runs.

**You should see:**
- `EXECUTE DBT PROJECT` returns success, exception, standard output, and an `OUTPUT_ARCHIVE_URL` for logs and artifacts.
- `snow dbt execute` streams run logs to stdout with full dbt timing and row counts.

> Docs: [EXECUTE DBT PROJECT](https://docs.snowflake.com/en/sql-reference/sql/execute-dbt-project) · [snow dbt execute](https://docs.snowflake.com/en/developer-guide/snowflake-cli/command-reference/dbt-commands/execute/overview)

In [ ]:
/* WALKTHROUGH ONLY — DO NOT RUN
-- Step 4.4a: Execute the full project via SQL (the form you embed in a Task)
-- Source: https://docs.snowflake.com/en/sql-reference/sql/execute-dbt-project
USE ROLE SYSADMIN;

EXECUTE DBT PROJECT RAW_DATA_TO_PRODUCTION_DASHBOARDS_HOL.PUBLIC.dbt_orders_project
  ARGS = 'build --target prod';

-- Run only the orders_curated model (dbt subcommand 'run' is always first in ARGS):
-- EXECUTE DBT PROJECT RAW_DATA_TO_PRODUCTION_DASHBOARDS_HOL.PUBLIC.dbt_orders_project
--   ARGS = 'run --select orders_curated --target prod';

-- Run dbt tests only (useful as a post-run Task in a chained pipeline):
-- EXECUTE DBT PROJECT RAW_DATA_TO_PRODUCTION_DASHBOARDS_HOL.PUBLIC.dbt_orders_project
--   ARGS = 'test --select orders_curated --target prod';

-- Step 4.4b: Equivalent snow CLI invocation (terminal, not Snowflake SQL)
-- Source: https://docs.snowflake.com/en/developer-guide/snowflake-cli/command-reference/dbt-commands/execute/overview
-- snow dbt execute dbt_orders_project build --target prod
END WALKTHROUGH */
SELECT 'Walkthrough only — no dbt project executed.' AS walkthrough_status;


### Step 4.5 — Schedule the project with a Snowflake task

Scheduling a dbt project is identical to scheduling any other Snowflake operation: wrap `EXECUTE DBT PROJECT` inside a native **Task**. No Airflow, no cron server, no external DAG runner. The Task triggers on a CRON schedule or on an upstream Task completing — the same event-driven chaining from the Streams+Tasks section applies here too, letting you sequence a `dbt run` Task followed by a `dbt test` Task in the same graph.

**Why it matters:** a team running `dbt run` on a VM or in GitHub Actions can move execution entirely into Snowflake, keep the dbt authoring experience unchanged, and schedule it alongside every other Snowflake workload in one system.

> Docs: [Schedule dbt project execution](https://docs.snowflake.com/en/developer-guide/dbt/dbt-execute) · [CREATE TASK](https://docs.snowflake.com/en/sql-reference/sql/create-task)

In [ ]:
/* WALKTHROUGH ONLY — DO NOT RUN
-- Step 4.5: Wrap execution in a native Snowflake Task (same pattern as Streams+Tasks section)
-- Source: https://docs.snowflake.com/en/sql-reference/sql/execute-dbt-project
USE ROLE SYSADMIN;

CREATE OR ALTER TASK RAW_DATA_TO_PRODUCTION_DASHBOARDS_HOL.PUBLIC.dbt_orders_daily_task
  WAREHOUSE = RAW_DATA_TO_PRODUCTION_DASHBOARDS_WH
  SCHEDULE = 'USING CRON 0 6 * * * UTC'       -- daily at 06:00 UTC
AS
  EXECUTE DBT PROJECT RAW_DATA_TO_PRODUCTION_DASHBOARDS_HOL.PUBLIC.dbt_orders_project
    ARGS = 'run --target prod';               -- dbt subcommand first; targets the prod profile in profiles.yml

-- Tasks start SUSPENDED by default — resume to activate
ALTER TASK RAW_DATA_TO_PRODUCTION_DASHBOARDS_HOL.PUBLIC.dbt_orders_daily_task RESUME;

-- Optional: chain a test task after the run task
-- CREATE OR ALTER TASK RAW_DATA_TO_PRODUCTION_DASHBOARDS_HOL.PUBLIC.dbt_orders_test_task
--   WAREHOUSE = RAW_DATA_TO_PRODUCTION_DASHBOARDS_WH
--   AFTER RAW_DATA_TO_PRODUCTION_DASHBOARDS_HOL.PUBLIC.dbt_orders_daily_task
-- AS
--   EXECUTE DBT PROJECT RAW_DATA_TO_PRODUCTION_DASHBOARDS_HOL.PUBLIC.dbt_orders_project
--     ARGS = 'test --target prod';

-- Verify the task is registered and scheduled
SHOW TASKS LIKE 'dbt_orders_daily_task%';
END WALKTHROUGH */
SELECT 'Walkthrough only — no task created or resumed.' AS walkthrough_status;


> **Apply on your account**
>
> **Applying this in production:**
> - Create a dedicated service role for the dbt project: SYSADMIN grants the role CREATE DBT PROJECT on the schema; the role becomes the project owner and is the `role` value in your `profiles.yml` target block.
> - Set `DEFAULT_TARGET = 'prod'` on the project object and maintain a second project object pointing at a staging branch with `DEFAULT_TARGET = 'dev'` for pre-prod validation.
> - **Deploying updated code:** on a mutable live-version object, push to git, run `ALTER GIT REPOSITORY ... FETCH;`, then `ALTER DBT PROJECT ... DEPLOY FROM '@stage/branches/main';`. During the 2026_06 transition, use the numbered-version path documented for accounts that have not migrated.
> - Add dbt test definitions in your model `.yml` files and invoke `dbt build` or `dbt test` when tests must run. A `dbt run` command does not run tests by itself.
> - **Hybrid pipeline:** your dbt model's `source()` macro references **RAW_ORDERS_TABLE** (Bronze from Section 1) — the exact same source Dynamic Tables and Streams+Tasks use. All three transformation patterns are composable on top of the same source data.

## Section 5 — BI Tool Connection + Workspaces Dashboard and Streamlit Path

**The problem:** Your analytics pipeline is built and refreshing — but there is no role configured for BI tool service accounts to authenticate through, and no live visual your stakeholders can open today.

Common BI tools connect to Snowflake through a
**[JDBC/ODBC driver](https://docs.snowflake.com/en/developer-guide/jdbc/jdbc)** or a
native Snowflake connector and an authenticating role. The same **RBAC** pattern taught
throughout this series applies here: `BI_SERVICE_ROLE` gets USAGE on the query warehouse,
database, and schema, plus SELECT on the Gold Dynamic Table. That is the documented
read-only path for this demo.

For the hands-on payoff, the Workspaces notebook uses the active Snowpark session and
Altair to render a KPI summary, chart, and data preview from `DT_ORDERS_ANALYTICS`.
New Notebooks in Workspaces do not embed Streamlit components. A persistent Streamlit
application is a separate app project that you create and deploy from the same Workspace.


> **Outcome:** put the finished pipeline in front of a stakeholder as a live dashboard.

### How BI tools connect to Snowflake

Common BI tools connect with a native Snowflake connector or a standard
**[JDBC](https://docs.snowflake.com/en/developer-guide/jdbc/jdbc)**/ODBC adapter.
A typical service connection needs three configured pieces:

| Piece | What it is |
|-------|-----------|
| **Driver** | JDBC/ODBC adapter or native Snowflake connector |
| **Role** | A minimal-privilege role the service account authenticates as |
| **Endpoint** | Account URL + warehouse + database/schema |

**Why it matters:** Rather than giving a BI tool personal credentials, you create a
dedicated service-account role with only the access it needs: warehouse compute,
schema navigation, and SELECT on specific tables. This isolates blast radius and keeps
audit logs readable — every query from the BI tool shows up in `QUERY_HISTORY` under
`BI_SERVICE_ROLE`, not under a human user.

> Docs: [Connecting to Snowflake with JDBC](https://docs.snowflake.com/en/developer-guide/jdbc/jdbc)

### Step 5.1 — Create the BI service role

Create `BI_SERVICE_ROLE` — the identity object that any BI tool service account
will authenticate as. We create the role first (identity), then assign data
privileges (access) in the next step, keeping the two concerns separate.

**Why it matters:** Separating role creation from privilege assignment is
least privilege in action: `USERADMIN` owns identity objects (roles and users), while
`SECURITYADMIN` controls what each role can reach. This means the security team can
audit and change BI access without touching the role definition itself.

**Run as:** This lab uses `USERADMIN`, the system role intended to create and manage
users and roles. A custom role with the global `CREATE ROLE` privilege can also do it.

> Docs: [CREATE ROLE](https://docs.snowflake.com/en/sql-reference/sql/create-role)

In [ ]:
-- Step 5.1: Create the BI service role
-- Source: https://docs.snowflake.com/en/sql-reference/sql/create-role
-- Created as ACCOUNTADMIN for the same reason as the Setup cell: if this role
-- already exists in your account under a different owning role, USERADMIN
-- cannot see it and CREATE ROLE IF NOT EXISTS fails.
USE ROLE ACCOUNTADMIN;

-- IF NOT EXISTS rather than drop-then-create: dropped roles cannot be recovered,
-- and Step 5.2's grants are idempotent, so a re-run converges either way.
CREATE ROLE IF NOT EXISTS BI_SERVICE_ROLE
  COMMENT = 'Minimal-privilege role for BI tool service accounts (Tableau, Power BI, Sigma, Looker, etc.)';

-- Normalize ownership to USERADMIN, the role that owns user and role objects and
-- the role the cleanup cell uses to drop this one.
-- Source: https://docs.snowflake.com/en/sql-reference/sql/grant-ownership
GRANT OWNERSHIP ON ROLE BI_SERVICE_ROLE
  TO ROLE USERADMIN COPY CURRENT GRANTS;

-- Confirm creation
USE ROLE USERADMIN;
SHOW ROLES LIKE 'BI_SERVICE_ROLE';


### Step 5.2 — Grant the minimal privilege set

This demo's read-only query path needs four object privileges:

| Privilege | Object | Why |
|-----------|--------|-----|
| USAGE | Warehouse | Provides compute to run queries |
| USAGE | Database | Allows navigation into the lab database |
| USAGE | Schema | Allows navigation into the PUBLIC schema |
| SELECT | Dynamic Table DT_ORDERS_ANALYTICS | Read-only access to the analytics result |

**Why it matters:** This is the least-privilege floor for a read-only BI connection.
No INSERT, CREATE, or DDL — if these credentials are ever compromised, the attacker
can only read one table.

Each grant names its object directly, so this step can be re-run at any point.

**Run as:** This lab uses `SECURITYADMIN`, which holds the global `MANAGE GRANTS`
privilege. In production, the object owner or another role with sufficient grant
authority can issue these grants.

> Docs: [GRANT privilege](https://docs.snowflake.com/en/sql-reference/sql/grant-privilege)

In [ ]:
-- Step 5.2: Grant the minimal privilege set to BI_SERVICE_ROLE
-- Source: https://docs.snowflake.com/en/sql-reference/sql/grant-privilege
USE ROLE SECURITYADMIN;

-- 1. Compute: USAGE on the lab warehouse
GRANT USAGE ON WAREHOUSE RAW_DATA_TO_PRODUCTION_DASHBOARDS_WH
  TO ROLE BI_SERVICE_ROLE;

-- 2. Data navigation: USAGE on database + schema
GRANT USAGE ON DATABASE RAW_DATA_TO_PRODUCTION_DASHBOARDS_HOL
  TO ROLE BI_SERVICE_ROLE;
GRANT USAGE ON SCHEMA RAW_DATA_TO_PRODUCTION_DASHBOARDS_HOL.PUBLIC
  TO ROLE BI_SERVICE_ROLE;

-- 3. Data access: SELECT on the dashboard-ready Dynamic Table only
GRANT SELECT ON DYNAMIC TABLE RAW_DATA_TO_PRODUCTION_DASHBOARDS_HOL.PUBLIC.DT_ORDERS_ANALYTICS
  TO ROLE BI_SERVICE_ROLE;


### Step 5.3 — Verify the privilege set

Confirm `BI_SERVICE_ROLE` holds exactly the four privileges you granted — no more,
no less.

**You should see:**
- Four rows: `USAGE` on the warehouse, `USAGE` on the database, `USAGE` on the
  schema, and `SELECT` on the `DYNAMIC_TABLE` object `DT_ORDERS_ANALYTICS`
- `granted_by` shows `SECURITYADMIN` for every row

> Docs: [SHOW GRANTS TO ROLE](https://docs.snowflake.com/en/sql-reference/sql/show-grants)

In [ ]:
-- Step 5.3: Verify the grant set (read-only confirm)
-- Source: https://docs.snowflake.com/en/sql-reference/sql/show-grants
SHOW GRANTS TO ROLE BI_SERVICE_ROLE;

---

## The payoff: a Workspaces-native dashboard

The pipeline is complete: raw rows flow through the medallion layers into
**DT_ORDERS_ANALYTICS**, a Dynamic Table that refreshes on schedule. Now we
surface it.

Notebooks in Workspaces support the active Snowpark session through
`get_active_session()` and use notebook-native libraries such as Altair for
visualization. Streamlit remains available in Workspaces as a separate app project:
create a `.py` app, preview it privately, and deploy it when it is ready to share.

**Why it matters:** A stakeholder doesn't know what a Dynamic Table is — but they
can read a dashboard. Closing the loop from raw data to a visual your business
owner can open is the definition of "production-ready."

> Docs: [Editing and running Notebooks in Workspaces](https://docs.snowflake.com/en/user-guide/ui-snowsight/notebooks-in-workspaces/notebooks-in-workspaces-edit-run) · [Streamlit in Snowflake in Workspaces](https://docs.snowflake.com/en/developer-guide/streamlit/streamlit-in-workspaces/streamlit-in-workspaces-overview)

### Step 5.4 — Build the notebook dashboard

The Python cell below renders a three-part dashboard directly in this notebook:

- **KPI summary** — rows displayed, total orders, total revenue, and freshest day
- **Altair chart** — daily revenue broken down by order status
- **Data preview** — the underlying rows so stakeholders can drill down

The query uses the active Snowpark session and fully qualifies the Gold Dynamic
Table, which is the recommended Workspaces notebook pattern.

**Why it matters:** You just went from a CSV on someone's laptop to a self-refreshing
notebook view backed by a live, governed pipeline. A separately deployed Streamlit
app is the production path when you need a persistent shared interface.

> Docs: [Editing and running Notebooks in Workspaces](https://docs.snowflake.com/en/user-guide/ui-snowsight/notebooks-in-workspaces/notebooks-in-workspaces-edit-run) · [Migrating legacy notebooks to Workspaces](https://docs.snowflake.com/en/user-guide/ui-snowsight/notebooks-in-workspaces/notebooks-in-workspaces-migrate)

In [ ]:
# Step 5.4 — Workspaces notebook dashboard with Altair
# Source: https://docs.snowflake.com/en/user-guide/ui-snowsight/notebooks-in-workspaces/notebooks-in-workspaces-edit-run
from snowflake.snowpark.context import get_active_session
from IPython.display import Markdown, display
import altair as alt
import pandas as pd

session = get_active_session()

# The grant-verification cell runs as SECURITYADMIN. Return to the object-owning
# lab role before querying the Gold Dynamic Table.
session.sql("USE ROLE SYSADMIN").collect()
session.sql("USE WAREHOUSE RAW_DATA_TO_PRODUCTION_DASHBOARDS_WH").collect()

# Gold is pre-aggregated, so load the complete result for accurate KPI totals.
df = session.sql(
    "SELECT * FROM RAW_DATA_TO_PRODUCTION_DASHBOARDS_HOL.PUBLIC.DT_ORDERS_ANALYTICS"
    " ORDER BY ORDER_DAY, STATUS"
).to_pandas()

if df.empty:
    raise ValueError("DT_ORDERS_ANALYTICS returned no rows; refresh the Dynamic Table first.")

df["ORDER_DAY"] = pd.to_datetime(df["ORDER_DAY"])
# Cast ORDER_COUNT explicitly: to_pandas() can narrow NUMBER(18,0) to a small
# integer dtype based on the values it sees, which is fragile as data grows.
total_orders = int(df["ORDER_COUNT"].astype("int64").sum())
total_revenue = float(df["TOTAL_REVENUE"].sum())

# Format the currency as a string. A bare float renders in scientific notation
# (7.650457e+09), which is unreadable as a headline KPI.
kpis = pd.DataFrame([{
    "Gold rows": f"{len(df):,}",
    "Total orders": f"{total_orders:,}",
    "Total revenue": f"${total_revenue:,.2f}",
    "Freshest order day": df["ORDER_DAY"].max().date().isoformat(),
}])

display(Markdown("### Orders Analytics Dashboard"))
display(Markdown("Source: `DT_ORDERS_ANALYTICS` · Gold Dynamic Table"))
display(kpis)

chart = (
    alt.Chart(df)
    .mark_bar()
    .encode(
        x=alt.X("ORDER_DAY:T", title="Order day"),
        y=alt.Y("sum(TOTAL_REVENUE):Q", title="Total revenue"),
        color=alt.Color("STATUS:N", title="Status"),
        tooltip=[
            alt.Tooltip("ORDER_DAY:T", title="Order day"),
            alt.Tooltip("STATUS:N", title="Status"),
            alt.Tooltip("TOTAL_REVENUE:Q", title="Revenue", format=",.2f"),
            alt.Tooltip("ORDER_COUNT:Q", title="Orders", format=","),
        ],
    )
    .properties(title="Daily revenue by status", width=760, height=320)
    .interactive()
)
display(chart)
display(Markdown("#### Underlying data"))
display(df.head(20))

### Verify BI_SERVICE_ROLE in Snowsight (read-only)

[Open in Snowsight](https://app.snowflake.com/_deeplink/#/account/roles/graph)

Read-only — confirm what you just built; all objects were created by the SQL above.

> **Apply on your account**
>
> **In production:**
> - Replace `BI_SERVICE_ROLE` with a tool-scoped name (e.g. `SIGMA_SERVICE_ROLE`,
>   `TABLEAU_SERVICE_ROLE`) so grants stay traceable per tool. Create one role per BI
>   tool.
> - Grant the role to the Snowflake user the BI tool authenticates as (run as
>   SECURITYADMIN): `GRANT ROLE BI_SERVICE_ROLE TO USER <bi_service_user>;`
> - Use key-pair authentication — not password — for service accounts:
>   `ALTER USER <bi_service_user> SET RSA_PUBLIC_KEY = '<public_key>';`
> - Keep Altair for notebook analysis. For a persistent shared interface, create a
>   Streamlit app in the Workspace and deploy it to a schema after granting the app
>   owner the documented Streamlit privileges. This lab does not deploy an app.
> - Use `GRANT SELECT ON FUTURE DYNAMIC TABLES IN SCHEMA … TO ROLE BI_SERVICE_ROLE;`
>   for future Gold Dynamic Tables. A `FUTURE TABLES` grant does not cover Dynamic Tables.

## Cleanup (disabled by default)

The teardown statements are commented out so a full run does not destroy what you
just built. Uncomment them when you want to remove the lab.

| Object | Dropped by |
|--------|-----------|
| `RAW_DATA_TO_PRODUCTION_DASHBOARDS_HOL` | `SYSADMIN` (owner) |
| `BI_SERVICE_ROLE` | `USERADMIN` (creator) |
| `RAW_DATA_TO_PRODUCTION_DASHBOARDS_WH` | `SYSADMIN` (owner) |

Dropping the database removes the tables, Dynamic Tables, stream, tasks, and logs
inside it. Run Step 3.10 first — the root task must be suspended before any task in
the graph can be dropped.

`SNOWFLAKE_SAMPLE_DATA` and the account-level `EXECUTE TASK` grant are left alone:
the share uses no storage, and both may predate this lab.

> Docs: [DROP TASK](https://docs.snowflake.com/en/sql-reference/sql/drop-task) ·
> [DROP DATABASE](https://docs.snowflake.com/en/sql-reference/sql/drop-database)


In [ ]:
-- Cleanup is disabled. Uncomment the six lines below to tear down the lab.

-- USE ROLE SYSADMIN;
-- DROP DATABASE IF EXISTS RAW_DATA_TO_PRODUCTION_DASHBOARDS_HOL;
-- USE ROLE USERADMIN;
-- DROP ROLE IF EXISTS BI_SERVICE_ROLE;
-- USE ROLE SYSADMIN;
-- DROP WAREHOUSE IF EXISTS RAW_DATA_TO_PRODUCTION_DASHBOARDS_WH;

SELECT 'Cleanup cell reached. Only the DROP statements uncommented above were run.'
    AS CLEANUP_STATUS;
